Zadaniem klasyfikacyjnym dla modeli było ustalenie liczby "1" nad przekątną w postaci Jordana.

Macierze losowe są generowane w następujący sposób:
* stwórz macierz $J$ z odpowiednią liczbą $1$ nad przekątną oraz z wartościami własnymi $\lambda = 1$ na przekątnej
* wylosuj macierz $S$, taką że $\kappa(S) < 10^5$
* $ X = S^{-1} J S$
* macierz jest normalizowana (względem normy Frobeniusa): $ X = 1/\|X\|_F X$

Losowanie macierzy $S$ odbywa się wg różnych metod:
* `random`: $S$ generowana przez `np.random.rand(d, d)`
* `upper`: $S$ jest górnotrójkątna, `S = np.triu(np.random.rand(d, d))`
* `int`: $S$ jest o wyrazach całkowitych z zakresu 1 do 100: `S = np.random.randint(0, 100, size=(d, d))`
* `ortho`: $S$ jest ortonormalna `A = np.random.rand(d,d) \ S, _ = np.linalg.qr(A)` 

Próbowane było generowanie zbioru treningowego i testowego wg różnych metod; rozmiar macierzy $5 \times 5$. 

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from jordanutils import *

rng = np.random.default_rng(seed=21)

In [2]:
import torch_directml

torch_directml.device()

device(type='privateuseone', index=0)

In [4]:
# ---- Define the PyTorch model ----
class SimpleNN(nn.Module):
    def __init__(self, d):
        super(SimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(d * d, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, d),
        )

    def forward(self, x):
        x = self.flatten(x)
        x = self.layers(x)
        return x


# ---- Training + Evaluation function ----
def train_and_test_model(train_mode, test_mode, verbose=1):
    # --- Device selection logic ---
    try:
        import torch_directml

        device = torch_directml.device()
        backend = "directml"
    except ImportError:
        if torch.cuda.is_available():
            device = torch.device("cuda")
            backend = "cuda"
        else:
            device = torch.device("cpu")
            backend = "cpu"

    if verbose:
        print(f"Using device: {device} (backend: {backend})")

    # --- Dataset setup ---
    d = 5
    dataset_size = 50000
    manager = LabelsManager([0], [1], [2], [3], [4], dataset_size=dataset_size)
    X, y = generate_testset(d, manager, mode=train_mode)
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=21
    )

    # --- Convert to tensors ---
    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    X_val = torch.tensor(X_val, dtype=torch.float32)
    y_val = torch.tensor(y_val, dtype=torch.long)

    # --- DataLoaders ---
    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

    # --- Model, Loss, Optimizer ---
    model = SimpleNN(d).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # --- Early Stopping setup ---
    best_val_acc = 0.0
    patience = 3
    counter = 0
    best_weights = None

    # --- Training Loop ---
    for epoch in range(50):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

        # --- Validation Loop ---
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb)
                pred_labels = preds.argmax(1)
                correct += (pred_labels == yb).sum().item()
                total += yb.size(0)
        val_acc = correct / total

        if verbose:
            print(f"Epoch {epoch+1:02d} - Val Acc: {val_acc:.4f}")

        # --- Early Stopping ---
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            counter = 0
            best_weights = model.state_dict()
        else:
            counter += 1
            if counter >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch+1}")
                break

    # --- Restore best model ---
    if best_weights is not None:
        model.load_state_dict(best_weights)

    # --- Testing ---
    manager.dataset_size = 1000
    X_test, y_test = generate_testset(d, manager, mode=test_mode)
    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(X_test)
        y_predicted = torch.argmax(outputs, dim=1).cpu().numpy()

    return y_test, y_predicted

In [ ]:
# Train: upper, test: random
y_test, y_pred = train_and_test_model("random", "random")
print(classification_report(y_test, y_pred))

Using device: privateuseone:0 (backend: directml)


c:\Users\micha\Documents\Studia\Magisterka\venv\Lib\site-packages\torch\optim\adam.py:534: UserWarning: The operator 'aten::lerp.Scalar_out' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  torch._foreach_lerp_(device_exp_avgs, device_grads, 1 - beta1)


Epoch 01 - Val Acc: 0.5931
Epoch 02 - Val Acc: 0.6318
Epoch 03 - Val Acc: 0.6378
Epoch 04 - Val Acc: 0.6423
Epoch 05 - Val Acc: 0.6598
Epoch 06 - Val Acc: 0.6671
Epoch 07 - Val Acc: 0.6724
Epoch 08 - Val Acc: 0.6693
Epoch 09 - Val Acc: 0.6718
Epoch 10 - Val Acc: 0.6776
Epoch 11 - Val Acc: 0.6863
Epoch 12 - Val Acc: 0.6880
Epoch 13 - Val Acc: 0.6874
Epoch 14 - Val Acc: 0.6951
Epoch 15 - Val Acc: 0.6942
Epoch 16 - Val Acc: 0.6937
Epoch 17 - Val Acc: 0.6993
Epoch 18 - Val Acc: 0.6996
Epoch 19 - Val Acc: 0.6965
Epoch 20 - Val Acc: 0.7020
Epoch 21 - Val Acc: 0.6989
Epoch 22 - Val Acc: 0.7013
Epoch 23 - Val Acc: 0.6929
Early stopping at epoch 23
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1000
           1       0.82      0.85      0.84      1000
           2       0.56      0.60      0.58      1000
           3       0.45      0.43      0.44      1000
           4       0.61      0.58      0.59      1000

    accuracy              

In [ ]:
# Train: int, test: random
y_test, y_pred = train_and_test_model("int", "random")
print(classification_report(y_test, y_pred))

c:\Users\micha\Documents\Studia\Magisterka\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1000
           1       0.83      0.88      0.85      1000
           2       0.64      0.52      0.57      1000
           3       0.44      0.42      0.43      1000
           4       0.58      0.69      0.63      1000

    accuracy                           0.70      5000
   macro avg       0.70      0.70      0.70      5000
weighted avg       0.70      0.70      0.70      5000



In [ ]:
# Train: random, test: upper
y_test, y_pred = train_and_test_model("random", "upper")
print(classification_report(y_test, y_pred))

c:\Users\micha\Documents\Studia\Magisterka\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
              precision    recall  f1-score   support

           0       0.95      1.00      0.98      1000
           1       0.58      0.66      0.62      1000
           2       0.35      0.45      0.39      1000
           3       0.27      0.18      0.22      1000
           4       0.62      0.51      0.56      1000

    accuracy                           0.56      5000
   macro avg       0.55      0.56      0.55      5000
weighted avg       0.55      0.56      0.55      5000



In [ ]:
# Train: random, test: int
y_test, y_pred = train_and_test_model("random", "int")
print(classification_report(y_test, y_pred))

c:\Users\micha\Documents\Studia\Magisterka\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1000
           1       0.83      0.85      0.84      1000
           2       0.63      0.54      0.58      1000
           3       0.44      0.40      0.42      1000
           4       0.56      0.68      0.61      1000

    accuracy                           0.69      5000
   macro avg       0.69      0.69      0.69      5000
weighted avg       0.69      0.69      0.69      5000



In [ ]:
# Train: random, test: orthonormal
y_test, y_pred = train_and_test_model("random", "ortho")
print(classification_report(y_test, y_pred))

c:\Users\micha\Documents\Studia\Magisterka\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1000
           1       0.62      1.00      0.77      1000
           2       0.33      0.57      0.42      1000
           3       0.24      0.15      0.18      1000
           4       1.00      0.05      0.10      1000

    accuracy                           0.55      5000
   macro avg       0.64      0.55      0.49      5000
weighted avg       0.64      0.55      0.49      5000



In [ ]:
# Train, test: orthonormal
y_test, y_pred = train_and_test_model("ortho", "ortho")
print(classification_report(y_test, y_pred))

c:\Users\micha\Documents\Studia\Magisterka\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1000
           1       1.00      1.00      1.00      1000
           2       1.00      1.00      1.00      1000
           3       1.00      1.00      1.00      1000
           4       1.00      1.00      1.00      1000

    accuracy                           1.00      5000
   macro avg       1.00      1.00      1.00      5000
weighted avg       1.00      1.00      1.00      5000



Widać, że nie ma żadnej różnicy między losowaniem z macierzy przejścia całkowitych czy rzeczywistych. Używanie macierzy przejścia górnotrójkątnych w obu przypadkach pogorszyło rezultaty. 

Model doskonale radzi sobie, gdy zawęzimy problem do sytuacji, gdy wektory własne/główne są do siebie ortogonalne (czego można było się spodziewać). Taką sytuację jednakże równie dobrze rozwiązuje obliczalny numerycznie rozkład Schura.

Z drugiej strony, wytrenowany na losowych danych model, słabo radzi sobie z rozróżnianiem wielkości bloków, gdy $S$ jest ortonormalna. 